# Step 1 check 2 - Jumps at the change of month in the climate imprint + QDM product

**Ad hoc test, outside the chain**: it reads the step-1 product of the supervisor's pipeline (`qdm/merged`), ERA5 at
0.25 deg and the raw EC-Earth3 inputs (`guido_inputs`), and writes only a cache on EOS
(`outputs/step1_check/cache/`) and figures / tables in `outputs/` next to this notebook.

**Question.** The QDM of the pipeline is fitted month by month (each month with its own transfer function, calibrated
on the month extended by 15 days on each side). Two consecutive months can therefore be corrected differently, which
would create a step in the daily series between the last day of a month and the first day of the next. Notebook 1
found, for tasmax, a domain-mean day-to-day change of 0.321 K on the first day of a month against 0.246 K on the other
days (ERA5: 0.234 / 0.236 K). Here the step is measured cell by cell and month by month.

**Method.** For each pixel, each month boundary b (last day L of a month, first day F of the next) and each year:
the change across the boundary, F - L, minus the mean of the four changes around it, (L-1) - (L-2), L - (L-1),
(F+1) - F, (F+2) - (F+1). Removing the neighbouring changes removes the seasonal tendency of these days, so that this
difference e has a mean of zero without a step. Over the years: the step s = mean of e (K, mm/day or probability),
its standard error SE = SD(e) / sqrt(n) and z = s / SE. Without a step z is a standard normal variable: about 1 % of
the (pixel, boundary) pairs have |z| > 2.58 and the root mean square of z is about 1. A deterministic step of the
monthly QDM appears as many more |z| > 2.58 and a root mean square of z well above 1, also in the run (2100-2149),
where there is no ERA5 to compare with.

Datasets: `ERA5_cal` (ERA5 0.25 deg, 1981-2014), `PROD_cal` and `PROD_run` (product at 0.25 deg, 1981-2014 and the
years of the run), `GCM_cal` and `GCM_run` (raw EC-Earth3 at 0.7 deg, no monthly processing: the baseline of a series
without steps). The Dec -> Jan boundary uses consecutive years only. pr: the step of the daily amount (mm/day) and of
the wet-day occurrence (>= `WET` mm/day). Domain: LISFLOOD domain without the land outside Europe
(`ba_utils.load_domain_mask`), all domain pixels, weights cos(lat).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
RUN = os.environ.get("STEP1_RUN", "stab2050nohos")          # run folder of the pipeline
BASE = Path(cfg["data_root"]) / "elisa" / "GCM_downscaling"
RUN_DIRS = BASE / "downscaling"                               # run folders created by guido_pipeline/setup_run.sh
INP = BASE / "guido_inputs"                                   # inputs written by guido_pipeline/prepare_inputs.sh
ALL_VARS = ["tasmax", "dtr", "pr"]
CAL = (1981, 2014)
WET = 1.0                                                     # mm/day, wet-day occurrence of pr
Z_CRIT = 2.576                                                # two-sided 1 %
OVERWRITE = False

MONTH_END = np.cumsum([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])   # day of the year of the last day of each month
MON = "Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split()
BOUND = [f"{MON[m]}-{MON[(m + 1) % 12]}" for m in range(12)]
CACHE = bu.out_dir(cfg, "step1_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
print(f"run {RUN} | cache {CACHE}")

## 1. Files, grids, domain, years of the run

In [ ]:
def product_file(var):
    run_dir = RUN_DIRS / f"climdown_cdo_{var}_europe_EC-Earth3-{RUN}_stab"
    fs = sorted(glob.glob(str(run_dir / "data" / "*" / "out" / "qdm" / "merged" / f"*01_12*{var}*.nc")))
    fs = [f for f in fs if "unique_qdm_" not in Path(f).name]      # monthly merges of postprocess_qdm, not the product
    return Path(fs[0]) if len(fs) == 1 else None


FILES = {v: {"ERA5": INP / "era5" / f"{v}_era5_1981_2014_europe.nc",
             "GCM": INP / RUN / f"{v}_EC-Earth3-{RUN}_HISTORICAL_europe.nc",
             "PROD": product_file(v)} for v in ALL_VARS}
VARS = [v for v in ALL_VARS if FILES[v]["PROD"] is not None and FILES[v]["ERA5"].exists() and FILES[v]["GCM"].exists()]
for v in ALL_VARS:
    print(f"{v:6s} " + " | ".join(f"{k}: {p.name if p is not None and p.exists() else 'MISSING'}" for k, p in FILES[v].items()))
print("variables checked:", VARS)
assert VARS, "no step-1 product yet"


def open_var(path, var):
    # daily field, standard coordinates (ascending lat, lon -180..180), singleton dimensions and 29 February dropped
    ds = xr.open_dataset(path, chunks={"time": 365}, **bu._TIME_KW)
    da = ds[var] if var in ds.data_vars else ds[list(ds.data_vars)[0]]
    for d in list(da.dims):
        if d not in ("time", "lat", "lon", "latitude", "longitude", "y", "x") and da.sizes[d] == 1:
            da = da.isel({d: 0}, drop=True)
    da = bu.norm_coords(da).transpose("time", "lat", "lon")
    t = da.indexes["time"]
    return da.isel(time=np.flatnonzero(~((t.month == 2) & (t.day == 29))))


F0 = open_var(FILES[VARS[0]]["ERA5"], VARS[0])
G0 = open_var(FILES[VARS[0]]["GCM"], VARS[0])
P0 = open_var(FILES[VARS[0]]["PROD"], VARS[0])
ry = np.unique(np.asarray(P0.indexes["time"].year)); ry = ry[ry > CAL[1]]
assert ry.size and (np.diff(ry) == 1).all(), f"years of the run not consecutive: {ry}"
RUNY = (int(ry[0]), int(ry[-1]))
COORD = {"fine": (F0.lat.values, F0.lon.values), "coarse": (G0.lat.values, G0.lon.values)}
DOM = {"fine": bu.load_domain_mask(cfg, "era5", like=F0.isel(time=0)).values,
       "coarse": bu.load_domain_mask(cfg, "ecearth3", like=G0.isel(time=0)).values}
DOMPIX = {g: np.flatnonzero(DOM[g].ravel()) for g in DOM}
WGT = {g: np.cos(np.deg2rad(np.repeat(COORD[g][0], COORD[g][1].size)))[DOMPIX[g]] for g in DOM}
DATASETS = {"ERA5_cal": ("ERA5", "fine", CAL), "PROD_cal": ("PROD", "fine", CAL), "PROD_run": ("PROD", "fine", RUNY),
            "GCM_cal": ("GCM", "coarse", CAL), "GCM_run": ("GCM", "coarse", RUNY)}
TAG = f"{RUN}_{CAL[0]}-{CAL[1]}_{RUNY[0]}-{RUNY[1]}"
print(f"run years {RUNY[0]}-{RUNY[1]} | domain: {DOMPIX['fine'].size} pixels at 0.25 deg, {DOMPIX['coarse'].size} cells at 0.7 deg")

## 2. Extraction (one year at a time, cached on EOS)

For each dataset, variable and kind (pr: amount and wet-day occurrence), sums over the years per boundary and domain
pixel: number of years, sum of e and of e squared, sum of |F - L| and of the mean |change| on the four days around.

In [ ]:
def kinds(var):
    return {"value": None, "wet": (lambda X: (X >= WET).astype("float64"))} if var == "pr" else {"value": None}


def extract(name, var):
    path = CACHE / f"jumps_{name}_{var}_{TAG}.npz"
    if path.exists() and not OVERWRITE:
        return dict(np.load(path))
    src, g, (y0, y1) = DATASETS[name]
    t0 = time.time()
    da = open_var(FILES[var][src], var)
    ref = G0 if src == "GCM" else F0
    assert bu.same_grid(da, ref), f"{name} {var}: not on the grid of {'EC-Earth3' if src == 'GCM' else 'ERA5'}"
    yrs = np.asarray(da.indexes["time"].year)
    K = kinds(var)
    acc = {f"{k}_{s}": np.zeros((12, DOMPIX[g].size)) for k in K for s in ("e", "e2", "a0", "a1")}
    n = np.zeros(12)
    tail = None
    for y in range(y0, y1 + 1):
        ti = np.flatnonzero(yrs == y)
        assert ti.size == 365, f"{name} {var}: {ti.size} days in {y}"
        X = da.isel(time=ti).values.reshape(365, -1)[:, DOMPIX[g]].astype("float64")
        wins = [(m, X[MONTH_END[m] - 3:MONTH_END[m] + 3]) for m in range(11)]   # days L-2 .. F+2
        if tail is not None:
            wins.append((11, np.vstack([tail, X[:3]])))                     # 29-31 December of y-1, 1-3 January of y
        tail = X[-3:]
        for m, w in wins:
            n[m] += 1
            for k, f in K.items():
                d = np.diff(w if f is None else f(w), axis=0)                 # 5 changes, d[2] across the boundary
                around = d[[0, 1, 3, 4]]
                e = d[2] - around.mean(0)
                acc[f"{k}_e"][m] += e; acc[f"{k}_e2"][m] += e ** 2
                acc[f"{k}_a0"][m] += np.abs(d[2]); acc[f"{k}_a1"][m] += np.abs(around).mean(0)
    out = {"n": n, **{k: v.astype("float32") for k, v in acc.items()}}
    np.savez(path, **out)
    print(f"  {name} {var}: {time.time() - t0:.0f} s")
    return out


E = {}
for var in VARS:
    for name in DATASETS:
        E[name, var] = extract(name, var)
    print(f"{var}: extracted")

## 3. Steps, z-scores and summary per dataset

In [ ]:
UNIT = {"tasmax": "K", "dtr": "K", "pr": "mm/day"}


def stats(name, var, kind):
    a, n = E[name, var], E[name, var]["n"][:, None]
    s = a[f"{kind}_e"] / n
    sd = np.sqrt(np.maximum(a[f"{kind}_e2"] / n - s ** 2, 0) * n / (n - 1))
    with np.errstate(invalid="ignore", divide="ignore"):
        z = s / (sd / np.sqrt(n))
    return s, z, a[f"{kind}_a0"].sum(0) / a[f"{kind}_a1"].sum(0)          # (12, pix), (12, pix), (pix)


def wmean(v, g):
    ok = np.isfinite(v)
    return float((WGT[g][ok] * v[ok]).sum() / WGT[g][ok].sum())


ROWS, BROWS, MAPS = [], [], {}
for var in VARS:
    for kind in kinds(var):
        unit = UNIT[var] if kind == "value" else "probability"
        for name, (src, g, yy) in DATASETS.items():
            s, z, ratio = stats(name, var, kind)
            rms_s = np.sqrt(np.nanmean(s ** 2, 0)); rms_z = np.sqrt(np.nanmean(z ** 2, 0))
            MAPS[var, kind, name] = rms_s
            ROWS.append(dict(var=var, kind=kind, unit=unit, dataset=name, years=f"{yy[0]}-{yy[1]}",
                             rms_step=wmean(rms_s, g), rms_z=wmean(rms_z, g),
                             share_abs_z_above_crit=100 * float((WGT[g] * (np.abs(z) > Z_CRIT).mean(0)).sum() / WGT[g].sum()),
                             ratio_abs_change=wmean(ratio, g)))
            for m in range(12):
                BROWS.append(dict(var=var, kind=kind, dataset=name, boundary=BOUND[m], mean_step=wmean(s[m], g),
                                  mean_abs_step=wmean(np.abs(s[m]), g),
                                  share_abs_z_above_crit=100 * float((WGT[g] * (np.abs(z[m]) > Z_CRIT)).sum() / WGT[g].sum())))
RES = pd.DataFrame(ROWS); BND = pd.DataFrame(BROWS)
RES.to_csv(OUT / f"month_jumps_{TAG}.csv", index=False); BND.to_csv(OUT / f"month_jumps_per_boundary_{TAG}.csv", index=False)
with pd.option_context("display.width", 220, "display.max_columns", None, "display.float_format", "{:.3f}".format):
    display(RES)
print("rms_step: root mean square of the step over the 12 boundaries (unit), domain mean | rms_z: the same for z (about 1 "
      f"without steps) | share_abs_z_above_crit: % of (pixel, boundary) with |z| > {Z_CRIT} (about 1 % without steps) | "
      "ratio_abs_change: mean |F - L| / mean |change| on the days around (about 1 without steps)")

## 4. Per boundary: domain-mean step of the product against ERA5 (1981-2014) and in the run

In [ ]:
for var in VARS:
    for kind in kinds(var):
        t = BND[(BND["var"] == var) & (BND["kind"] == kind)]
        tab = pd.concat({c: t.pivot(index="boundary", columns="dataset", values=c).loc[BOUND, list(DATASETS)]
                         for c in ("mean_step", "share_abs_z_above_crit")}, axis=1)
        print(f"\n{var} ({kind}): mean step (signed, domain mean) and % of pixels with |z| > {Z_CRIT}")
        with pd.option_context("display.width", 260, "display.max_columns", None, "display.float_format", "{:.3f}".format):
            display(tab)

fig, axes = plt.subplots(1, sum(len(kinds(v)) for v in VARS), figsize=(6 * sum(len(kinds(v)) for v in VARS), 3.8), squeeze=False)
i = 0
for var in VARS:
    for kind in kinds(var):
        ax = axes[0, i]; i += 1
        t = BND[(BND["var"] == var) & (BND["kind"] == kind)]
        for name, st in zip(DATASETS, ("k-", "C3-", "C3--", "C0-", "C0--")):
            ax.plot(range(12), t[t["dataset"] == name].set_index("boundary").loc[BOUND, "share_abs_z_above_crit"], st, marker="o", ms=3, label=name)
        ax.axhline(1, c="0.6", lw=0.8); ax.set_xticks(range(12)); ax.set_xticklabels(BOUND, rotation=60, fontsize=8)
        ax.set_ylabel(f"% of pixels with |z| > {Z_CRIT}"); ax.set_title(f"{var} ({kind})", fontsize=9)
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / f"month_jumps_per_boundary_{TAG}.png", dpi=110); plt.show()

## 5. Maps: root mean square of the step over the 12 boundaries

In [ ]:
BORDERS = bu.borders(cfg, "10M")
FLAT, FLON = COORD["fine"]
panels = [(var, kind) for var in VARS for kind in kinds(var)]
fig, axes = plt.subplots(len(panels), 3, figsize=(19, 4 * len(panels)), squeeze=False)
for i, (var, kind) in enumerate(panels):
    vmax = np.nanpercentile(MAPS[var, kind, "PROD_cal"], 98)
    for j, name in enumerate(("ERA5_cal", "PROD_cal", "PROD_run")):
        f = np.full(FLAT.size * FLON.size, np.nan); f[DOMPIX["fine"]] = MAPS[var, kind, name]
        ax = axes[i, j]
        pm = ax.pcolormesh(FLON, FLAT, f.reshape(FLAT.size, FLON.size), shading="nearest", cmap="viridis", vmin=0, vmax=vmax)
        bu.draw_borders(ax, BORDERS, lw=0.3)
        ax.set_title(f"{var} ({kind}) {name}: RMS step over the boundaries", fontsize=9)
        plt.colorbar(pm, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(OUT / f"month_jumps_maps_{TAG}.png", dpi=110); plt.show()

## 6. Reading

- `ERA5_cal` and the raw GCM (`GCM_cal`, `GCM_run`) give the level of a series without steps: rms_z about 1, about 1 %
  of |z| > 2.58, ratio_abs_change about 1. Their rms_step is the sampling noise of the step for that number of years.
- `PROD_cal` and `PROD_run` above that level: the monthly QDM creates steps. Their size (rms_step, in the unit of the
  variable) is to be compared with the day-to-day variability and with what matters for LISFLOOD (snowmelt and
  evaporation for the temperature, wet-day sequences for pr). Boundaries with the largest steps: section 4.
- A step present in the calibration years but not in the run (or the reverse) points to the part of the QDM that
  differs between the periods (the delta factors of the run); a step in both comes from the monthly calibration
  against ERA5.